In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%load_ext line_profiler
%load_ext memory_profiler


In [3]:
import corc.utils
import numpy as np
# import jax.numpy as jnp
# import jax.scipy.special
import corc.bhc.prior as prior
import time
import corc.bhc.bhc
import sklearn.model_selection
import corc.bhc.prior
import tqdm
import networkx as nx

In [4]:

def analyze_result(result, subsampled_data, subsampled_ys):
    arc_list = result.arc_list
    node_ids = result.node_ids

    G = nx.DiGraph()
    for arc in arc_list:
        G.add_edge(arc.source, arc.target)

    # Subgraph excluding last 6 nodes
    num_classes = len(np.unique(subsampled_ys))
    nodes_to_remove = node_ids[-(num_classes - 1) :]
    subgraph = G.subgraph([node for node in G.nodes() if node not in nodes_to_remove])
    subgraph = subgraph.to_undirected()
    # Connected components
    connected_components = list(nx.connected_components(subgraph))

    # Create y_pred labels
    y_pred = np.zeros(len(node_ids), dtype=int)
    for i, component in enumerate(connected_components):
        for node in component:
            y_pred[node] = i

    # Filter y_pred and y_true to only include nodes within subsampled_data range
    valid_indices = np.where(np.array(node_ids) < len(subsampled_data))[0]
    y_pred_filtered = y_pred[valid_indices]
    y_true_filtered = subsampled_ys[valid_indices]

    # print(np.unique(y_pred_filtered, return_counts=True)[1])
    # print(np.unique(y_true_filtered, return_counts=True)[1])

    # ARI calculation
    ari = sklearn.metrics.adjusted_rand_score(y_true_filtered, y_pred_filtered)

    return ari

In [ ]:
size = 200
X,y,tsne = corc.utils.load_dataset("densired_soft_8", cache_path="../../../cache")
if X.shape[0] > size and size > 0:
    _, X, _, y = sklearn.model_selection.train_test_split(
        X, y, test_size=size, stratify=y, random_state=42
    )

In [6]:
def get_bhc(subsampled_data, g=20, scale_factor=0.001, alpha=1):
    model = prior.NormalInverseWishart.create(subsampled_data, g, scale_factor)
    start_time = time.time()
    bhc_result = corc.bhc.bhc.BayesianHierarchicalClustering(
        subsampled_data, model, alpha, cut_allowed=False, verbose=True
    ).build()
    end_time = time.time()
    # print(f"Time taken for BHC: {end_time - start_time:.2f} seconds")
    return bhc_result


bhc_results = dict()
for i in tqdm.trange(10):
    # g = 5 * i + 10
    scale_factor = 0.001 * (i**2 + 1)
    bhc_results[scale_factor] = get_bhc(X, scale_factor=scale_factor)

for index, bhc_result in bhc_results.items():
    print(f"ARI for g={index} is {analyze_result(bhc_result, X, y):.2f}")

densired8
ARI for g=10 is 0.88
ARI for g=15 is 0.90
ARI for g=20 is 0.85
ARI for g=25 is 0.92
ARI for g=30 is 0.90
ARI for g=35 is 0.93
ARI for g=40 is 0.92
ARI for g=45 is 0.78
ARI for g=50 is 0.51
ARI for g=55 is 0.83

In [ ]:
# <!-- jax_X = jnp.array(X) -->
get_bhc(X)

(2000, 2000)
Time taken pairwise log_p numpy calculations: 79.33 seconds


In [7]:
%mprun -f corc.bhc.bhc.BayesianHierarchicalClustering.build get_bhc(X)


(200, 200)
Time taken pairwise log_p numpy calculations: 0.08 seconds
Time taken for initial pairwise calculations: 13.21 seconds


Merging clusters: 100%|██████████| 199/199 [00:24<00:00,  8.01it/s]

Time taken for merging: 24.85 seconds
Time taken for new comparisons: 24.29 seconds



Filename: /mnt/vast-nhr/projects/nim00012/git_martin/tneb_clustering/src/corc/bhc/bhc.py

Line #    Mem usage    Increment  Occurrences   Line Contents
    30    341.5 MiB    341.5 MiB           1       def build(self):
    31    341.5 MiB      0.0 MiB           1           n_objects = self.data.shape[0]
    32                                         
    33    341.5 MiB      0.0 MiB           1           weights = []
    34                                         
    35                                                 # active nodes
    36    341.5 MiB      0.0 MiB           1           active_nodes = np.arange(n_objects)
    37                                                 # assignments - starting each point in its own cluster
    38    341.5 MiB      0.0 MiB           1           assignments = np.arange(n_objects)
    39                                                 # stores information from temporary merges
    40    341.5 MiB      0.0 MiB           1           tmp_merge = None

In [8]:
# %lprun -f corc.bhc.bhc.BayesianHierarchicalClustering.build -f corc.bhc.prior.NormalInverseWishart.calc_log_mlh get_bhc(X)
# %lprun -f corc.bhc.prior.NormalInverseWishart.calc_log_mlh -f corc.bhc.prior.NormalInverseWishart._NormalInverseWishart__calc_posterior -f corc.bhc.prior.NormalInverseWishart._NormalInverseWishart__calc_log_prior get_bhc(X)
%lprun -f corc.bhc.bhc.BayesianHierarchicalClustering.build get_bhc(X)

(200, 200)
Time taken pairwise log_p numpy calculations: 0.08 seconds
Time taken for initial pairwise calculations: 0.40 seconds


Merging clusters: 100%|██████████| 199/199 [00:04<00:00, 43.51it/s] 

Time taken for merging: 4.58 seconds
Time taken for new comparisons: 4.48 seconds


Timer unit: 1e-09 s

Total time: 4.94676 s
File: /mnt/vast-nhr/projects/nim00012/git_martin/tneb_clustering/src/corc/bhc/bhc.py
Function: BayesianHierarchicalClustering.build at line 30

Line #      Hits         Time  Per Hit   % Time  Line Contents
    30                                               def build(self):
    31         1       2074.0   2074.0      0.0          n_objects = self.data.shape[0]
    32                                           
    33         1        300.0    300.0      0.0          weights = []
    34                                           
    35                                                   # active nodes
    36         1       3717.0   3717.0      0.0          active_nodes = np.arange(n_objects)
    37                                                   # assignments - starting each point in its own cluster
    38         1       1092.0   1092.0      0.0          assignments = np.arange(n_objects)
    39                                               

Time taken for initial pairwise calculations: 7.13 seconds
Time taken for merging: 12.36 seconds
Time taken for new comparisons: 12.32 seconds
Time taken for BHC: 19.65 seconds